# P2 Integrated Engineering Blueprint - Cell-by-Cell QA and Visualization

목적:

1. 오늘 생성된 `p2_2`, `p2_3`, `p2_4`, `p2_5`, `P2_6` 대표 IPYNB를 복제하고 hash로 검증한다.
2. active mart, strict mart, sample/split, source catalog, P5/P6 result artifact를 셀별로 검정한다.
3. 모델을 새로 학습하지 않고 기존 artifact의 수치를 읽어 해석한다.
4. 각 검정 블록에 대응하는 시각화 방안을 별도 표와 PNG로 저장한다.

금지:

- 신규 OLS/Ridge/GAM/XGBoost 학습
- bootstrap 재실행
- 원본 파일 수정
- 기존 readiness 폴더 덮어쓰기


## Cell 1. Setup / Run Provenance


In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import platform
import re
import shutil
import subprocess
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 3085
PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")
OUT_ROOT = PROJECT_ROOT / "workbook/p2/p2_integrated_engineering_blueprint_v1"

def run_cmd(args: list[str]) -> tuple[int, str]:
    try:
        out = subprocess.check_output(args, cwd=PROJECT_ROOT, stderr=subprocess.STDOUT, text=True)
        return 0, out.strip()
    except subprocess.CalledProcessError as exc:
        return exc.returncode, exc.output.strip()
    except Exception as exc:
        return 99, repr(exc)

git_code, git_full = run_cmd(["git", "rev-parse", "HEAD"])
_, git_short = run_cmd(["git", "rev-parse", "--short", "HEAD"])
_, git_status = run_cmd(["git", "status", "--short"])
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + f"_{git_short or 'nogit'}"
RUN_DIR = OUT_ROOT / "cell_runs" / RUN_ID
CLONE_DIR = RUN_DIR / "cloned_notebooks"
QA_DIR = RUN_DIR / "qa"
REPORT_DIR = RUN_DIR / "reports"
LOG_DIR = RUN_DIR / "logs"
FIG_DIR = RUN_DIR / "figures"
ART_DIR = RUN_DIR / "artifacts"
for directory in [CLONE_DIR, QA_DIR, REPORT_DIR, LOG_DIR, FIG_DIR, ART_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ENV = {
    "run_id": RUN_ID,
    "utc_started_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "output_run_dir": str(RUN_DIR),
    "git_commit_full": git_full if git_code == 0 else "",
    "git_commit_short": git_short,
    "git_dirty": bool(git_status.strip()),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "random_state": RANDOM_STATE,
}

print(json.dumps(ENV, ensure_ascii=False, indent=2))


{
  "run_id": "20260713T091641Z_5b1a3d5",
  "utc_started_at": "2026-07-13T09:16:41+00:00",
  "project_root": "/home/sieg/projects-wsl/SBS_dataScience",
  "output_run_dir": "/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5",
  "git_commit_full": "5b1a3d54266d881a839ad9a3cec750da66e94bc7",
  "git_commit_short": "5b1a3d5",
  "git_dirty": true,
  "python": "3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]",
  "platform": "Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39",
  "pandas": "3.0.3",
  "numpy": "2.5.0",
  "random_state": 3085
}


## Cell 2. Helper Functions


In [2]:
def sha256_file(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def read_csv_safe(path: Path) -> pd.DataFrame:
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, low_memory=False)


def load_table(path: Path) -> tuple[pd.DataFrame | None, str, str | None]:
    if not path.exists():
        return None, "missing", None
    try:
        if path.suffix.lower() == ".parquet":
            return pd.read_parquet(path), "parquet", None
        if path.suffix.lower() == ".csv":
            return read_csv_safe(path), "csv", None
        if path.suffix.lower() == ".json":
            data = json.loads(path.read_text(encoding="utf-8"))
            if isinstance(data, (dict, list)):
                return pd.json_normalize(data), "json_normalize", None
            return pd.DataFrame({"value": [data]}), "json_scalar", None
        if path.suffix.lower() in {".xlsx", ".xls"}:
            xls = pd.ExcelFile(path)
            return pd.DataFrame(
                [{"sheet": s, "rows": pd.read_excel(path, sheet_name=s).shape[0], "columns": pd.read_excel(path, sheet_name=s).shape[1]} for s in xls.sheet_names]
            ), "excel_sheet_inventory", None
    except Exception as exc:
        return None, "load_error", repr(exc)
    return None, "unprofiled", None


def shape_string(path: Path) -> tuple[str, str, str | None]:
    df, loader, error = load_table(path)
    if df is None:
        return "", loader, error
    return f"{df.shape[0]} x {df.shape[1]}", loader, error


def notebook_meta(path: Path) -> dict[str, Any]:
    row = {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
        "sha256": sha256_file(path),
        "cell_n": None,
        "code_cell_n": None,
        "markdown_cell_n": None,
        "executed_code_cell_n": None,
        "error_output_n": None,
        "stored_output_n": None,
        "parse_status": "MISSING" if not path.exists() else "UNREAD",
    }
    if not path.exists():
        return row
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
        cells = nb.get("cells", [])
        row.update(
            {
                "cell_n": len(cells),
                "code_cell_n": sum(c.get("cell_type") == "code" for c in cells),
                "markdown_cell_n": sum(c.get("cell_type") == "markdown" for c in cells),
                "executed_code_cell_n": sum(c.get("cell_type") == "code" and c.get("execution_count") is not None for c in cells),
                "error_output_n": sum(1 for c in cells for o in c.get("outputs", []) if o.get("output_type") == "error"),
                "stored_output_n": sum(len(c.get("outputs", [])) for c in cells if c.get("cell_type") == "code"),
                "parse_status": "PASS",
            }
        )
    except Exception as exc:
        row["parse_status"] = f"FAIL:{type(exc).__name__}:{exc}"
    return row


def atomic_csv(df: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
    tmp.replace(path)


def atomic_text(text: str, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def md_table(df: pd.DataFrame, cols: list[str] | None = None, max_rows: int | None = None) -> str:
    if cols is None:
        cols = list(df.columns)
    use = df[cols].copy() if len(df) else pd.DataFrame(columns=cols)
    if max_rows is not None:
        use = use.head(max_rows)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for _, row in use.iterrows():
        vals = []
        for col in cols:
            val = row.get(col, "")
            txt = "" if pd.isna(val) else (f"{val:.6g}" if isinstance(val, float) else str(val))
            vals.append(txt.replace("\n", " ").replace("|", "\\|"))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def pct_range_violations(df: pd.DataFrame) -> int:
    total = 0
    for col in [c for c in df.columns if c.endswith("_pct") or c.endswith("비율")]:
        vals = pd.to_numeric(df[col], errors="coerce")
        total += int(((vals < 0) | (vals > 100)).sum())
    return total


def save_fig(fig, filename: str) -> Path:
    path = FIG_DIR / filename
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def parse_bootstrap_ci(report_path: Path) -> str:
    if not report_path.exists():
        return ""
    text = report_path.read_text(encoding="utf-8")
    match = re.search(r"bootstrap residual D 95% CI:\s*`\[([^\]]+)\]%", text)
    return match.group(1) if match else ""


print("helper functions ready")


helper functions ready


## Cell 3. Representative Notebook Selection and Clone Audit


In [3]:
REPRESENTATIVE_NOTEBOOKS = [
    {"stage": "P2_2", "label": "P2_G0_source_eda", "path": "workbook/p2/p2_2/final/eda/P2_G0.ipynb", "reason": "초기 원천 EDA와 파일 구조 확인"},
    {"stage": "P2_2", "label": "P2_G1_kedi_structure", "path": "workbook/p2/p2_2/final/eda/P2_G1_kedi.ipynb", "reason": "KEDI 고등교육 구조자료 검사"},
    {"stage": "P2_2", "label": "P2_G1_concat_outcome", "path": "workbook/p2/p2_2/final/eda/P2_G1_concat_eda.ipynb", "reason": "성적·입결·취업·진학 outcome spine 결합"},
    {"stage": "P2_2", "label": "P2_G2_visual_admission", "path": "workbook/p2/p2_2/final/eda/P2_G2_정시입결_visual_eda.ipynb", "reason": "입결 proxy와 정시 입결 시각 검사"},
    {"stage": "P2_3", "label": "P3_1_handoff_builder", "path": "workbook/p2/p2_3/p3_1.ipynb", "reason": "D01-D05와 D08 handoff 구성"},
    {"stage": "P2_3", "label": "P3_2_goms_context", "path": "workbook/p2/p2_3/p3_2.ipynb", "reason": "GOMS D06/D07 노동시장 context 구성"},
    {"stage": "P2_4", "label": "P4_source_parquet_csv_eda", "path": "workbook/p2/p2_4/p2_G4_source_parquet_csv_eda.ipynb", "reason": "handoff parquet/csv 전체 source inspection"},
    {"stage": "P2_4", "label": "P4_dataset_cell_inspection", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/p2_G4_dataset_cell_inspection.ipynb", "reason": "dataset-by-dataset inspection"},
    {"stage": "P2_4", "label": "P4_contract_cleaning_1", "path": "workbook/p2/p2_4/p2_G4_1.ipynb", "reason": "P4 계약·무결성 검증"},
    {"stage": "P2_4", "label": "P4_contract_cleaning_2", "path": "workbook/p2/p2_4/p2_G4_2.ipynb", "reason": "P4 readiness 후속 검증"},
    {"stage": "P2_4", "label": "P2_grade_formation_strict", "path": "workbook/p2/p2_4/p2_grade_formation_v1_strict/p2_grade_formation_strict.ipynb", "reason": "strict P2 grade formation branch"},
    {"stage": "P2_4", "label": "P3_grade_residual_strict", "path": "workbook/p2/p2_4/p3_grade_residual_v1_strict/p3_grade_residual_strict.ipynb", "reason": "strict P3 residual handoff"},
    {"stage": "P2_4", "label": "P4_grade_signal_outcomes_strict", "path": "workbook/p2/p2_4/p4_grade_signal_outcomes_v1_strict/p4_grade_signal_outcomes_strict.ipynb", "reason": "strict P4 grade-signal/outcome analysis"},
    {"stage": "P2_5", "label": "P5_major7_heterogeneity_strict", "path": "workbook/p2/p2_5/p2_g5_1_strict.ipynb", "reason": "P5 major7 heterogeneity strict result"},
    {"stage": "P2_5", "label": "P5_A2_visual_summary", "path": "workbook/p2/p2_5/P2_G5_A2.ipynb", "reason": "P5 visual/numeric summary"},
    {"stage": "P2_6", "label": "P6_confirmatory_closure", "path": "workbook/p2/P2_6/p4_confirmatory_closure.ipynb", "reason": "P4 confirmatory closure"},
    {"stage": "P2_6", "label": "P6_strict_chain_runup", "path": "workbook/p2/P2_6/P2_G6_1.ipynb", "reason": "P3->P4->P5/P6 strict chain run-up"},
]

rows = []
for item in REPRESENTATIVE_NOTEBOOKS:
    src = PROJECT_ROOT / item["path"]
    meta = notebook_meta(src)
    clone_path = CLONE_DIR / item["path"]
    clone_path.parent.mkdir(parents=True, exist_ok=True)
    clone_status = "MISSING"
    if src.exists():
        shutil.copy2(src, clone_path)
        clone_status = "COPIED"
    row = {**item, **meta}
    row["clone_path"] = str(clone_path)
    row["clone_status"] = clone_status
    row["clone_sha256"] = sha256_file(clone_path) if clone_path.exists() else None
    row["clone_hash_match"] = bool(row["sha256"] and row["sha256"] == row["clone_sha256"])
    rows.append(row)

notebook_inventory = pd.DataFrame(rows)
atomic_csv(notebook_inventory, QA_DIR / "cell03_representative_notebook_clone_inventory.csv")
display(notebook_inventory[["stage", "label", "cell_n", "code_cell_n", "executed_code_cell_n", "error_output_n", "clone_status", "clone_hash_match"]])

assert notebook_inventory["exists"].all(), "대표 notebook 중 누락 파일 있음"
assert notebook_inventory["clone_hash_match"].all(), "복제 notebook hash mismatch 있음"
print("CELL03 PASS: 17 representative notebooks cloned with hash_match=1")


,stage,label,cell_n,code_cell_n,executed_code_cell_n,error_output_n,clone_status,clone_hash_match
0,P2_2,P2_G0_source_eda,16,14,14,0,COPIED,True
1,P2_2,P2_G1_kedi_structure,98,49,9,0,COPIED,True
2,P2_2,P2_G1_concat_outcome,1,1,1,0,COPIED,True
3,P2_2,P2_G2_visual_admission,3,1,1,0,COPIED,True
4,P2_3,P3_1_handoff_builder,15,12,12,0,COPIED,True
5,P2_3,P3_2_goms_context,45,43,43,0,COPIED,True
6,P2_4,P4_source_parquet_csv_eda,200,103,42,0,COPIED,True
7,P2_4,P4_dataset_cell_inspection,37,19,0,0,COPIED,True
8,P2_4,P4_contract_cleaning_1,12,6,6,0,COPIED,True
9,P2_4,P4_contract_cleaning_2,2,1,1,0,COPIED,True


CELL03 PASS: 17 representative notebooks cloned with hash_match=1


## Cell 4. Key File Inventory and Source Provenance


In [4]:
KEY_FILES = [
    {"label": "D08_active_mart", "path": "workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet", "role": "active 2024 151-column mart"},
    {"label": "D08_clean_integrity_mart", "path": "workbook/p2/p2_4/p4_preprocessing_integrity_v1/data/clean_department_model_base_2024.parquet", "role": "preprocessing/integrity clean mart"},
    {"label": "D08_strict_drop_mart", "path": "workbook/p2/p2_4/source_eda/strict_clean_v1/mart_department_model_base_2024_strict_drop.parquet", "role": "strict model input mart"},
    {"label": "source_catalog", "path": "workbook/p2/p2_4/p4_preprocessing_integrity_v1/qa/data_source_catalog.csv", "role": "source provenance catalog"},
    {"label": "source_file_inventory", "path": "workbook/p2/p2_4/p4_preprocessing_integrity_v1/qa/data_source_file_inventory.csv", "role": "source file inventory"},
    {"label": "column_registry_v4", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_registry_v4.csv", "role": "151-column role contract"},
    {"label": "phase_sample_registry_final", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv", "role": "phase sample registry"},
    {"label": "phase_sample_membership_final", "path": "workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet", "role": "row-level sample masks"},
    {"label": "dim_school_split", "path": "workbook/p2/p2_3/p4_handoff_candidate/shared/dim_school_split.csv", "role": "school split"},
    {"label": "p5_strict_status", "path": "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/reports/P5_2024_MAJOR7_HETEROGENEITY_V2_STRICT_STATUS.json", "role": "P5 strict status"},
    {"label": "p5_strict_insight", "path": "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/artifacts/P5_STRICT_INSIGHT_SUMMARY.csv", "role": "P5 AME summary"},
    {"label": "p5_difference", "path": "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/artifacts/P5_TABLE3_PROGRESSION_MINUS_EMPLOYMENT_AME.csv", "role": "P5 progression-employment differences"},
    {"label": "p6_status", "path": "workbook/p2/P2_6/qa/P2_G6_1_STATUS_TABLE.csv", "role": "P6 status"},
    {"label": "p6_key_numbers", "path": "workbook/p2/P2_6/artifacts/P2_G6_1_KEY_NUMBERS.csv", "role": "P6 key numbers"},
    {"label": "p6_report", "path": "workbook/p2/P2_6/reports/P2_G6_1_STRICT_CHAIN_RUNUP_REPORT.md", "role": "P6 strict chain report"},
]

file_rows = []
for item in KEY_FILES:
    path = PROJECT_ROOT / item["path"]
    shape, loader, error = shape_string(path)
    file_rows.append(
        {
            **item,
            "absolute_path": str(path),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else None,
            "shape": shape,
            "loader": loader,
            "load_error": error,
            "sha256": sha256_file(path),
        }
    )
file_inventory = pd.DataFrame(file_rows)
atomic_csv(file_inventory, QA_DIR / "cell04_key_file_inventory.csv")
display(file_inventory[["label", "role", "exists", "shape", "sha256"]])

source_catalog = read_csv_safe(PROJECT_ROOT / "workbook/p2/p2_4/p4_preprocessing_integrity_v1/qa/data_source_catalog.csv")
atomic_csv(source_catalog, QA_DIR / "cell04_source_catalog_copy.csv")
display(source_catalog[["catalog_id", "official_source", "official_url", "local_source_files", "final_column_blocks"]])

assert file_inventory["exists"].all(), "핵심 파일 누락"
assert len(source_catalog) >= 8, "source catalog row 수 부족"
print("CELL04 PASS: key files and source provenance catalog loaded")


,label,role,exists,shape,sha256
0,D08_active_mart,active 2024 151-column mart,True,10242 x 151,598b68b31b5358dfd23839d4c138cc64838d05876b7791...
1,D08_clean_integrity_mart,preprocessing/integrity clean mart,True,10242 x 184,dc4d320a44c2e6a109e4da58c8ee3aefaf98572e99d53f...
2,D08_strict_drop_mart,strict model input mart,True,7592 x 151,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...
3,source_catalog,source provenance catalog,True,9 x 13,524555d9ddfad44908fac25969d9732d410bc212c2089f...
4,source_file_inventory,source file inventory,True,30 x 11,67f79d5ba4b62edc7231c8a67063730ace686aabba7f23...
5,column_registry_v4,151-column role contract,True,151 x 27,c8f3954654de2853c92c4643b7ccf5cb271a7ad50de925...
6,phase_sample_registry_final,phase sample registry,True,12 x 12,d518bab363399a31037590a2cb285f2a76e88318f5d25b...
7,phase_sample_membership_final,row-level sample masks,True,7592 x 21,c6504fd04c918ce33439462f7b277aa8ae4363f185ef93...
8,dim_school_split,school split,True,200 x 6,85c2c851ddcfd02d5ead41dbd9424124e4ef1993347842...
9,p5_strict_status,P5 strict status,True,1 x 39,419d18bb0b47f44e52b81ee25d9973b7a3e98cb3915f95...


,catalog_id,official_source,official_url,local_source_files,final_column_blocks
0,SRC01_KEDI_STRUCTURE,KEDI 교육통계서비스 / 고등교육통계,https://kess.kedi.re.kr/,KEDI_raw_excel,"S0, B, DENOM, K, QUALITY"
1,SRC02_ACADEMYINFO_GRADE,대학알리미 대학정보공시,https://www.academyinfo.go.kr/,grade_raw,"GRADE, QUALITY"
2,SRC03_ACADEMYINFO_EMPLOYMENT,대학알리미 대학정보공시,https://www.academyinfo.go.kr/,employment_raw,"EMP, QUALITY"
3,SRC04_ACADEMYINFO_PROGRESSION,대학알리미 대학정보공시,https://www.academyinfo.go.kr/,progression_raw,"PROG, QUALITY"
4,SRC05_ADIGA_ADMISSION,대입정보포털 어디가,https://www.adiga.kr/,adiga_seed; adiga_registry; adiga_raw; adiga_d...,"Q, QUALITY"
5,SRC06_CREDIT_FORFEIT_POLICY,"local curated policy workbook, likely universi...",https://www.academyinfo.go.kr/,policy_raw,"POLICY, QUALITY"
6,SRC07_MAJOR_CONTEXT_WAGE_COMPANY_CERT,계열 단위 임금/기업유형/자격증 context local contract,https://www.academyinfo.go.kr/,wage_reference; wage_quartile; wage_contract; ...,"C24, QUALITY"
7,SRC08_KEIS_GOMS_LABOR_CONTEXT,한국고용정보원 고용조사분석시스템 / 대졸자직업이동경로조사(GOMS),https://survey.keis.or.kr/,goms_distribution_long; goms_continuous_long; ...,"GOMS, QUALITY"
8,SRC09_INTERNAL_BRIDGES_SPLITS_REGISTRIES,internal deterministic bridge/registry artifacts,NaN,bridge_school_alias; bridge_campus_alias; brid...,"K, S0, QUALITY"


CELL04 PASS: key files and source provenance catalog loaded


## Cell 5. Active Mart Key / Grain / Range Integrity


In [5]:
D08_PATH = PROJECT_ROOT / "workbook/p2/p2_3/p4_handoff_candidate/shared/mart_department_model_base_2024.parquet"
CLEAN_PATH = PROJECT_ROOT / "workbook/p2/p2_4/p4_preprocessing_integrity_v1/data/clean_department_model_base_2024.parquet"
STRICT_PATH = PROJECT_ROOT / "workbook/p2/p2_4/source_eda/strict_clean_v1/mart_department_model_base_2024_strict_drop.parquet"

d08 = pd.read_parquet(D08_PATH)
clean = pd.read_parquet(CLEAN_PATH)
strict = pd.read_parquet(STRICT_PATH)

target_cols = ["a_rate_pct", "health_employment_rate_pct", "graduate_school_progression_rate_pct"]
mart_audit_rows = []
for dataset, df, path in [("D08_active", d08, D08_PATH), ("D08_clean", clean, CLEAN_PATH), ("D08_strict", strict, STRICT_PATH)]:
    row = {
        "dataset": dataset,
        "shape": f"{df.shape[0]} x {df.shape[1]}",
        "sha256": sha256_file(path),
        "outcome_row_id_null_n": int(df["outcome_row_id"].isna().sum()) if "outcome_row_id" in df.columns else None,
        "outcome_row_id_duplicate_rows": int(df.duplicated("outcome_row_id").sum()) if "outcome_row_id" in df.columns else None,
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "school_n": int(df["school_uid"].nunique()) if "school_uid" in df.columns else None,
        "campus_n": int(df["campus_uid"].nunique()) if "campus_uid" in df.columns else None,
        "dept_n": int(df["dept_uid"].nunique()) if "dept_uid" in df.columns else None,
        "major7_non_null_n": int(df["major_group_7"].notna().sum()) if "major_group_7" in df.columns else None,
        "major7_missing_n": int(df["major_group_7"].isna().sum()) if "major_group_7" in df.columns else None,
        "pct_range_violation_n": pct_range_violations(df),
    }
    for target in target_cols:
        row[f"{target}_non_null_n"] = int(df[target].notna().sum()) if target in df.columns else None
    mart_audit_rows.append(row)

mart_integrity = pd.DataFrame(mart_audit_rows)
atomic_csv(mart_integrity, QA_DIR / "cell05_mart_key_grain_range_integrity.csv")
display(mart_integrity)

assert int(d08["outcome_row_id"].isna().sum()) == 0
assert int(d08.duplicated("outcome_row_id").sum()) == 0
assert int(d08.duplicated().sum()) == 0
assert pct_range_violations(d08) == 0
print("CELL05 PASS: active D08 key, exact duplicate, and pct range checks passed")


,dataset,shape,sha256,outcome_row_id_null_n,outcome_row_id_duplicate_rows,exact_duplicate_rows,school_n,campus_n,dept_n,major7_non_null_n,major7_missing_n,pct_range_violation_n,a_rate_pct_non_null_n,health_employment_rate_pct_non_null_n,graduate_school_progression_rate_pct_non_null_n
0,D08_active,10242 x 151,598b68b31b5358dfd23839d4c138cc64838d05876b7791...,0,0,0,200,452,10222,10099,143,0,10242,7477,7587
1,D08_clean,10242 x 184,dc4d320a44c2e6a109e4da58c8ee3aefaf98572e99d53f...,0,0,0,200,452,10222,10099,143,0,10242,7477,7587
2,D08_strict,7592 x 151,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,0,0,0,197,261,7592,7592,0,0,7592,5600,5674


CELL05 PASS: active D08 key, exact duplicate, and pct range checks passed


## Cell 6. Schema / Role / Feature Block Contract


In [6]:
registry_path = PROJECT_ROOT / "workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/department_model_column_registry_v4.csv"
registry = read_csv_safe(registry_path)

registry_audit = pd.DataFrame(
    [
        {"metric": "registry_shape_rows", "value": registry.shape[0]},
        {"metric": "registry_shape_columns", "value": registry.shape[1]},
        {"metric": "duplicate_column_rows", "value": int(registry.duplicated("column").sum())},
        {"metric": "unresolved_other_count", "value": int(registry.get("feature_block", pd.Series(dtype=str)).eq("OTHER").sum())},
        {"metric": "missing_feature_block_count", "value": int(registry.get("feature_block", pd.Series(dtype=str)).isna().sum())},
        {"metric": "active_identifier_feature_count", "value": int(((registry.get("model_default_active", False) == True) & (registry.get("is_identifier", False) == True)).sum())},
    ]
)
block_counts = registry["feature_block"].value_counts(dropna=False).rename_axis("feature_block").reset_index(name="column_n")
source_counts = registry["source_dataset"].value_counts(dropna=False).rename_axis("source_dataset").reset_index(name="column_n")
source_block = registry.groupby(["source_dataset", "feature_block"], dropna=False).size().reset_index(name="column_n")

atomic_csv(registry_audit, QA_DIR / "cell06_registry_audit.csv")
atomic_csv(block_counts, QA_DIR / "cell06_feature_block_counts.csv")
atomic_csv(source_counts, QA_DIR / "cell06_source_dataset_counts.csv")
atomic_csv(source_block, QA_DIR / "cell06_source_block_counts.csv")

display(registry_audit)
display(block_counts)
display(source_counts)

assert int(registry.duplicated("column").sum()) == 0
assert int(registry["feature_block"].eq("OTHER").sum()) == 0
print("CELL06 PASS: registry has no duplicate column and no OTHER role")


,metric,value
0,registry_shape_rows,151
1,registry_shape_columns,27
2,duplicate_column_rows,0
3,unresolved_other_count,0
4,missing_feature_block_count,0
5,active_identifier_feature_count,0


,feature_block,column_n
0,QUALITY,33
1,GOMS,28
2,K,27
3,DENOM,15
4,C24,15
5,S0,12
6,PROG,6
7,B,5
8,Q,4
9,GRADE,3


,source_dataset,column_n
0,D01_v2,51
1,D03_v2,32
2,D07_HANDOFF,28
3,D02_v2,25
4,D04_v2,15


CELL06 PASS: registry has no duplicate column and no OTHER role


## Cell 7. Sample / Split Integrity


In [7]:
sample_registry_path = PROJECT_ROOT / "workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv"
sample_membership_path = PROJECT_ROOT / "workbook/p2/p2_4/p4_modeling_readiness_v4/data/P4_PHASE_SAMPLE_MEMBERSHIP_FINAL.parquet"
split_path = PROJECT_ROOT / "workbook/p2/p2_3/p4_handoff_candidate/shared/dim_school_split.csv"

sample_registry = read_csv_safe(sample_registry_path)
sample_membership = pd.read_parquet(sample_membership_path)
split_df = read_csv_safe(split_path)

split_summary = split_df["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="school_n")
flag_cols = [c for c in sample_membership.columns if c.endswith("_READY") or c.startswith("P")]
membership_counts = []
for col in flag_cols:
    vals = sample_membership[col]
    membership_counts.append(
        {
            "sample_id": col,
            "true_n": int(vals.fillna(False).astype(bool).sum()) if vals.dtype != object else int(vals.astype(str).str.lower().eq("true").sum()),
            "na_n": int(vals.isna().sum()),
        }
    )
membership_counts = pd.DataFrame(membership_counts)

atomic_csv(sample_registry, QA_DIR / "cell07_phase_sample_registry.csv")
atomic_csv(split_summary, QA_DIR / "cell07_school_split_summary.csv")
atomic_csv(membership_counts, QA_DIR / "cell07_membership_flag_counts.csv")

display(sample_registry[[c for c in ["sample_id", "row_n", "school_n", "train_n", "validation_n", "test_n"] if c in sample_registry.columns]])
display(split_summary)
display(membership_counts.head(20))

assert split_df["school_uid"].nunique() == split_df.shape[0]
assert set(split_df["split"].dropna().unique()).issubset({"train", "val", "validation", "test"})
print("CELL07 PASS: school split has unique schools and valid split labels")


,sample_id,row_n,school_n,train_n,validation_n,test_n
0,P1_STRUCTURE_READY,5600,185,4080,871,649
1,P1_SELECTIVITY_READY,2355,130,1742,360,253
2,P2_STRUCTURE_READY,7592,197,5534,1168,890
3,P2_SELECTIVITY_READY,3119,146,2293,514,312
4,P3_STRUCTURE_READY,7592,197,5534,1168,890
5,P3_SELECTIVITY_READY,3119,146,2293,514,312
6,P4_E_STRUCTURE_READY,5600,185,4080,871,649
7,P4_P_STRUCTURE_READY,5674,194,4129,884,661
8,P4_JOINT_STRUCTURE_READY,5600,185,4080,871,649
9,P4_E_SELECTIVITY_READY,2355,130,1742,360,253


,split,school_n
0,train,140
1,test,30
2,val,30


,sample_id,true_n,na_n
0,sample_P1_STRUCTURE_READY,5600,0
1,sample_P1_SELECTIVITY_READY,2355,0
2,sample_P2_STRUCTURE_READY,7592,0
3,sample_P2_SELECTIVITY_READY,3119,0
4,sample_P3_STRUCTURE_READY,7592,0
5,sample_P3_SELECTIVITY_READY,3119,0
6,sample_P4_E_STRUCTURE_READY,5600,0
7,sample_P4_P_STRUCTURE_READY,5674,0
8,sample_P4_JOINT_STRUCTURE_READY,5600,0
9,sample_P4_E_SELECTIVITY_READY,2355,0


CELL07 PASS: school split has unique schools and valid split labels


## Cell 8. Target Distribution and Boundary Checks


In [8]:
target_profile_rows = []
for dataset, df in [("D08_active", d08), ("D08_strict", strict)]:
    for target in target_cols:
        vals = pd.to_numeric(df[target], errors="coerce")
        target_profile_rows.append(
            {
                "dataset": dataset,
                "target": target,
                "n": int(vals.notna().sum()),
                "missing_n": int(vals.isna().sum()),
                "mean": float(vals.mean()),
                "std": float(vals.std()),
                "min": float(vals.min()),
                "p01": float(vals.quantile(0.01)),
                "p05": float(vals.quantile(0.05)),
                "p25": float(vals.quantile(0.25)),
                "median": float(vals.median()),
                "p75": float(vals.quantile(0.75)),
                "p95": float(vals.quantile(0.95)),
                "p99": float(vals.quantile(0.99)),
                "max": float(vals.max()),
                "zero_n": int((vals == 0).sum()),
                "hundred_n": int((vals == 100).sum()),
                "range_violation_n": int(((vals < 0) | (vals > 100)).sum()),
            }
        )
target_profile = pd.DataFrame(target_profile_rows)
atomic_csv(target_profile, QA_DIR / "cell08_target_distribution_profile.csv")
display(target_profile)

assert target_profile["range_violation_n"].sum() == 0
assert target_profile.loc[(target_profile["dataset"] == "D08_active") & (target_profile["target"] == "a_rate_pct"), "n"].iloc[0] == 10242
print("CELL08 PASS: target distributions computed and bounded 0-100")


,dataset,target,n,missing_n,mean,std,min,p01,p05,p25,median,p75,p95,p99,max,zero_n,hundred_n,range_violation_n
0,D08_active,a_rate_pct,10242,0,41.726425,17.011000,0.0,0.000000,14.643038,32.476539,39.791550,49.940473,73.036091,92.857140,100.0,319,79,0
1,D08_active,health_employment_rate_pct,7477,2765,52.675671,19.217381,0.0,0.000000,20.000000,41.176472,52.941177,64.000000,86.206894,100.000000,100.0,136,180,0
2,D08_active,graduate_school_progression_rate_pct,7587,2655,8.062091,13.011699,0.0,0.000000,0.000000,0.000000,2.380952,10.841194,35.483871,57.192856,100.0,3343,8,0
3,D08_strict,a_rate_pct,7592,0,41.664440,14.373115,0.0,3.556818,22.824521,32.893457,39.572113,48.571430,69.223560,83.991481,100.0,67,19,0
4,D08_strict,health_employment_rate_pct,5600,1992,52.694118,17.513512,0.0,11.111111,23.076923,41.666668,52.941177,63.636364,82.223259,96.512032,100.0,20,41,0
5,D08_strict,graduate_school_progression_rate_pct,5674,1918,8.254618,12.895537,0.0,0.000000,0.000000,0.000000,2.777778,11.111111,35.778389,57.746964,100.0,2342,2,0


CELL08 PASS: target distributions computed and bounded 0-100


## Cell 9. Missingness / Domain / Context Grain Checks


In [9]:
missing_rows = []
for col in d08.columns:
    missing_rows.append(
        {
            "column": col,
            "dtype": str(d08[col].dtype),
            "missing_n": int(d08[col].isna().sum()),
            "missing_pct": float(d08[col].isna().mean() * 100),
            "nunique": int(d08[col].nunique(dropna=True)),
        }
    )
missing_profile = pd.DataFrame(missing_rows).sort_values(["missing_pct", "missing_n"], ascending=False)

context_cols = registry.loc[registry["feature_block"].isin(["C24", "GOMS"]), "column"].dropna().tolist()
context_grain_rows = []
for col in context_cols:
    if col not in d08.columns:
        continue
    unique_by_major = d08.groupby("major_group_7", dropna=False)[col].nunique(dropna=True)
    context_grain_rows.append(
        {
            "column": col,
            "feature_block": registry.loc[registry["column"].eq(col), "feature_block"].iloc[0],
            "max_unique_within_major7": int(unique_by_major.max()) if len(unique_by_major) else None,
            "non_null_n": int(d08[col].notna().sum()),
            "status": "PASS_MAJOR_CONTEXT_CONSTANT" if len(unique_by_major) and unique_by_major.max() <= 1 else "WARN_VARIES_WITHIN_MAJOR7",
        }
    )
context_grain = pd.DataFrame(context_grain_rows)

atomic_csv(missing_profile, QA_DIR / "cell09_missingness_profile.csv")
atomic_csv(context_grain, QA_DIR / "cell09_context_grain_check.csv")
display(missing_profile.head(30))
display(context_grain.head(40))

context_warn_n = int(context_grain["status"].ne("PASS_MAJOR_CONTEXT_CONSTANT").sum()) if len(context_grain) else 0
print(f"CELL09 STATUS: context_warn_n={context_warn_n}; top missing columns saved")


,column,dtype,missing_n,missing_pct,nunique
121,ctx24_industry_top3_pct,Float32,10242,100.000000,0
122,ctx24_industry_hhi,Float32,10242,100.000000,0
14,selectivity_proxy_pct,Float32,6505,63.512986,1609
87,competition_ratio,Float32,4985,48.672134,3947
88,admission_yield_ratio,Float32,4985,48.672134,842
89,admit_per_applicant_ratio,Float32,4982,48.642843,3926
93,student_faculty_ratio,Float32,3863,37.717243,3288
94,fulltime_faculty_share_pct,Float32,3863,37.717243,613
18,employment_rate_pct,Float32,2765,26.996680,1135
19,health_employment_rate_pct,Float32,2765,26.996680,1196


,column,feature_block,max_unique_within_major7,non_null_n,status
0,ctx24_reference_sample_n,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
1,ctx24_mean_income_10kkrw,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
2,ctx24_median_income_10kkrw,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
3,ctx24_income_300plus_pct,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
4,ctx24_income_400plus_pct,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
5,ctx24_large_company_pct,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
6,ctx24_mid_company_pct,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
7,ctx24_small_company_pct,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
8,ctx24_large_mid_company_pct,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT
9,ctx24_public_nonprofit_pct,C24,1,10099,PASS_MAJOR_CONTEXT_CONSTANT


CELL09 STATUS: context_warn_n=0; top missing columns saved


## Cell 10. Existing P5/P6 Numeric Result Readout


In [10]:
p6_numbers_path = PROJECT_ROOT / "workbook/p2/P2_6/artifacts/P2_G6_1_KEY_NUMBERS.csv"
p6_status_path = PROJECT_ROOT / "workbook/p2/P2_6/qa/P2_G6_1_STATUS_TABLE.csv"
p6_report_path = PROJECT_ROOT / "workbook/p2/P2_6/reports/P2_G6_1_STRICT_CHAIN_RUNUP_REPORT.md"
p5_status_path = PROJECT_ROOT / "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/reports/P5_2024_MAJOR7_HETEROGENEITY_V2_STRICT_STATUS.json"
p5_insight_path = PROJECT_ROOT / "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/artifacts/P5_STRICT_INSIGHT_SUMMARY.csv"
p5_diff_path = PROJECT_ROOT / "workbook/p2/p2_5/p5_major7_heterogeneity_v2_strict/artifacts/P5_TABLE3_PROGRESSION_MINUS_EMPLOYMENT_AME.csv"

p6_numbers = read_csv_safe(p6_numbers_path)
p6_status = read_csv_safe(p6_status_path)
p5_insight = read_csv_safe(p5_insight_path)
p5_diff = read_csv_safe(p5_diff_path)
with p5_status_path.open(encoding="utf-8") as f:
    p5_status_json = json.load(f)

p6_metric = {str(r.metric): float(r.value) for r in p6_numbers.itertuples(index=False) if pd.notna(r.value)}
bootstrap_ci = parse_bootstrap_ci(p6_report_path)

result_rows = []
for metric, value in p6_metric.items():
    unit = p6_numbers.loc[p6_numbers["metric"].eq(metric), "unit"].iloc[0]
    result_rows.append({"source": "P2_6", "metric": metric, "value": value, "unit": unit})
for _, r in p5_diff.iterrows():
    result_rows.append({"source": "P2_5", "metric": f"P5_DIFF_{r['major_group_7']}", "value": float(r["difference"]) * 100, "unit": "percentage point"})
result_readout = pd.DataFrame(result_rows)

interpretation = pd.DataFrame(
    [
        {
            "topic": "P3-S grade residual",
            "observation": f"OOF R2={p6_metric.get('P3_FULL_OOF_R2', np.nan):.3f}, locked test R2={p6_metric.get('P3_FULL_TEST_R2', np.nan):.3f}, OOF MAE={p6_metric.get('P3_FULL_OOF_MAE', np.nan):.3f}",
            "interpretation": "구조+정책 기반 A비율 예측력은 제한적이다.",
            "limit": "이 셀은 기존 artifact를 읽고 해석할 뿐 재학습하지 않는다.",
        },
        {
            "topic": "P4 raw A signal",
            "observation": f"RAW_A employment={p6_metric.get('P4_RAW_A_EMPLOYMENT_AME_PP', np.nan):.3f}pp, progression={p6_metric.get('P4_RAW_A_PROGRESSION_AME_PP', np.nan):.3f}pp",
            "interpretation": "기존 artifact 기준 A비율 10pp 관련 신호는 대학원 진학 쪽 수치가 더 크다.",
            "limit": "인과효과가 아니라 조건부 association readout이다.",
        },
        {
            "topic": "P4 residual signal",
            "observation": f"resid employment={p6_metric.get('P4_RESID_EMPLOYMENT_AME_PP', np.nan):.3f}pp, resid progression={p6_metric.get('P4_RESID_PROGRESSION_AME_PP', np.nan):.3f}pp, D={p6_metric.get('P4_RESID_D_PP', np.nan):.3f}pp, CI=[{bootstrap_ci}]pp",
            "interpretation": "OOF residual readout에서도 진학 쪽 추가 신호가 더 크다.",
            "limit": "P2_6 보고서는 bootstrap approximation 경고를 둔다.",
        },
    ]
)
top_major = p5_diff.assign(diff_pp=lambda x: x["difference"].astype(float) * 100).sort_values("diff_pp", ascending=False).head(7)

atomic_csv(result_readout, QA_DIR / "cell10_existing_numeric_result_readout.csv")
atomic_csv(interpretation, QA_DIR / "cell10_interpretation_scaffold.csv")
atomic_csv(top_major, QA_DIR / "cell10_p5_major7_difference_pp.csv")

display(result_readout)
display(interpretation)
display(top_major)
print("CELL10 PASS: existing P5/P6 artifacts read; no model fit executed")


,source,metric,value,unit
0,P2_6,P3_FULL_OOF_MAE,10.473046,a_rate_pct point
1,P2_6,P3_FULL_OOF_R2,0.056153,R2
2,P2_6,P3_FULL_TEST_R2,-0.066762,R2
3,P2_6,P3_FULL_RAW_RESIDUAL_CORR,0.926507,correlation
4,P2_6,P3_FULL_RESIDUAL_TO_RAW_VARIANCE_RATIO,0.943843,ratio
5,P2_6,P4_RAW_A_EMPLOYMENT_AME_PP,0.616300,percentage point
6,P2_6,P4_RAW_A_PROGRESSION_AME_PP,1.725986,percentage point
7,P2_6,P4_RESID_EMPLOYMENT_AME_PP,0.635722,percentage point
8,P2_6,P4_RESID_PROGRESSION_AME_PP,1.657726,percentage point
9,P2_6,P4_RESID_D_PP,1.022005,percentage point


,topic,observation,interpretation,limit
0,P3-S grade residual,"OOF R2=0.056, locked test R2=-0.067, OOF MAE=1...",구조+정책 기반 A비율 예측력은 제한적이다.,이 셀은 기존 artifact를 읽고 해석할 뿐 재학습하지 않는다.
1,P4 raw A signal,"RAW_A employment=0.616pp, progression=1.726pp",기존 artifact 기준 A비율 10pp 관련 신호는 대학원 진학 쪽 수치가 더 크다.,인과효과가 아니라 조건부 association readout이다.
2,P4 residual signal,"resid employment=0.636pp, resid progression=1....",OOF residual readout에서도 진학 쪽 추가 신호가 더 크다.,P2_6 보고서는 bootstrap approximation 경고를 둔다.


,major_group_7,employment_ame,employment_ci_low,employment_ci_high,employment_n,progression_ame,progression_ci_low,progression_ci_high,progression_n,difference,diff_pp
5,NAT,-0.004780,-0.018358,0.008953,583,0.036730,0.011689,0.069876,583,0.041510,4.151033
2,ENG,0.019294,0.008278,0.028755,911,0.045698,0.016228,0.091214,911,0.026404,2.640438
0,ART,-0.004299,-0.014682,0.007325,646,0.006174,0.000515,0.016800,646,0.010473,1.047337
4,MED,-0.005657,-0.031390,0.011755,374,0.003801,-0.000593,0.073071,374,0.009458,0.945773
6,SOC,0.007144,-0.002409,0.016208,848,0.011690,0.005174,0.023299,848,0.004546,0.454583
3,HUM,0.020456,0.009140,0.031212,535,0.013732,0.004090,0.031646,535,-0.006724,-0.672409
1,EDU,0.015633,0.000256,0.031026,602,0.005316,0.001640,0.012117,602,-0.010318,-1.031796


CELL10 PASS: existing P5/P6 artifacts read; no model fit executed


## Cell 11. Concrete Visualization Plan


In [11]:
visual_plan = pd.DataFrame(
    [
        {
            "viz_id": "VIZ01_SOURCE_AND_NOTEBOOK_LINEAGE",
            "question": "대표 notebook와 핵심 파일이 어떤 단계 흐름으로 연결되는가?",
            "data": "representative_notebook_clone_inventory + key_file_inventory",
            "plot": "stage별 notebook count/clone status + key file shape/SHA table",
            "encoding": "x=stage, y=count, color=clone_status; table includes shape/SHA",
            "output": "figures/cell12_lineage_inventory.png",
            "interpretation_rule": "clone hash mismatch가 있으면 해당 단계는 재현성 검토 대상",
        },
        {
            "viz_id": "VIZ02_SAMPLE_TARGET_DASHBOARD",
            "question": "phase별 sample 크기와 target 관측 N은 충분한가?",
            "data": "phase_sample_registry + target_profile",
            "plot": "sample row_n horizontal bar + target non-null bar",
            "encoding": "sample_id on y, row_n on x; target on x, non-null N on y",
            "output": "figures/cell12_sample_target_dashboard.png",
            "interpretation_rule": "selectivity sample이 구조 sample보다 작으면 Q branch 일반화 한계로 표시",
        },
        {
            "viz_id": "VIZ03_TARGET_DISTRIBUTIONS",
            "question": "A/취업/진학 target 분포는 0/100 boundary와 skew를 얼마나 갖는가?",
            "data": "D08 active mart",
            "plot": "3-panel histogram with median line and zero/hundred annotations",
            "encoding": "x=rate_pct, y=row count, facet=target",
            "output": "figures/cell13_target_distributions.png",
            "interpretation_rule": "zero mass가 큰 progression은 two-part sensitivity 필요성을 설명",
        },
        {
            "viz_id": "VIZ04_FEATURE_SIGNAL_CORRELATION",
            "question": "모델 재학습 없이 주요 구조/Q/context 변수가 target과 어떤 단순 association을 보이는가?",
            "data": "D08 numeric columns selected by registry blocks S0/B/Q/C24/GOMS",
            "plot": "target별 top Spearman absolute correlation heat/bar",
            "encoding": "feature on y, rho on x, color=target",
            "output": "figures/cell14_feature_signal_correlation.png",
            "interpretation_rule": "상관은 feature signal 가설일 뿐 feature selection/인과근거가 아님",
        },
        {
            "viz_id": "VIZ05_P5_P6_RESULT_READOUT",
            "question": "기존 P5/P6 artifact가 저장한 결과 신호는 어떻게 요약되는가?",
            "data": "P2_6 key numbers + P5 major7 difference",
            "plot": "P4 AME bar + P5 major7 progression-employment difference bar",
            "encoding": "AME percentage point on y/x; major_group_7 on y",
            "output": "figures/cell15_result_readout.png",
            "interpretation_rule": "기존 결과 읽기이며 이 notebook에서 추정한 신규 효과가 아님",
        },
    ]
)
atomic_csv(visual_plan, ART_DIR / "cell11_visualization_plan.csv")
atomic_text(md_table(visual_plan), REPORT_DIR / "CELL_BY_CELL_VISUALIZATION_PLAN.md")
display(visual_plan)
print("CELL11 PASS: concrete visualization plan saved")


,viz_id,question,data,plot,encoding,output,interpretation_rule
0,VIZ01_SOURCE_AND_NOTEBOOK_LINEAGE,대표 notebook와 핵심 파일이 어떤 단계 흐름으로 연결되는가?,representative_notebook_clone_inventory + key_...,stage별 notebook count/clone status + key file ...,"x=stage, y=count, color=clone_status; table in...",figures/cell12_lineage_inventory.png,clone hash mismatch가 있으면 해당 단계는 재현성 검토 대상
1,VIZ02_SAMPLE_TARGET_DASHBOARD,phase별 sample 크기와 target 관측 N은 충분한가?,phase_sample_registry + target_profile,sample row_n horizontal bar + target non-null bar,"sample_id on y, row_n on x; target on x, non-n...",figures/cell12_sample_target_dashboard.png,selectivity sample이 구조 sample보다 작으면 Q branch 일...
2,VIZ03_TARGET_DISTRIBUTIONS,A/취업/진학 target 분포는 0/100 boundary와 skew를 얼마나 갖는가?,D08 active mart,3-panel histogram with median line and zero/hu...,"x=rate_pct, y=row count, facet=target",figures/cell13_target_distributions.png,zero mass가 큰 progression은 two-part sensitivity...
3,VIZ04_FEATURE_SIGNAL_CORRELATION,모델 재학습 없이 주요 구조/Q/context 변수가 target과 어떤 단순 as...,D08 numeric columns selected by registry block...,target별 top Spearman absolute correlation heat...,"feature on y, rho on x, color=target",figures/cell14_feature_signal_correlation.png,상관은 feature signal 가설일 뿐 feature selection/인과근...
4,VIZ05_P5_P6_RESULT_READOUT,기존 P5/P6 artifact가 저장한 결과 신호는 어떻게 요약되는가?,P2_6 key numbers + P5 major7 difference,P4 AME bar + P5 major7 progression-employment ...,AME percentage point on y/x; major_group_7 on y,figures/cell15_result_readout.png,기존 결과 읽기이며 이 notebook에서 추정한 신규 효과가 아님


CELL11 PASS: concrete visualization plan saved


## Cell 12. Visualization 01-02: Lineage, Sample, Target Availability


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
stage_counts = notebook_inventory.groupby(["stage", "clone_status"]).size().reset_index(name="notebook_n")
pivot = stage_counts.pivot(index="stage", columns="clone_status", values="notebook_n").fillna(0)
pivot.plot(kind="bar", stacked=True, ax=axes[0], color=["#3b6ea8", "#c44e52"])
axes[0].set_title("Representative notebook clones by stage")
axes[0].set_xlabel("stage")
axes[0].set_ylabel("notebook count")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))

sample_plot = sample_registry[["sample_id", "row_n"]].copy()
axes[1].barh(sample_plot["sample_id"], sample_plot["row_n"], color="#5c8a43")
axes[1].invert_yaxis()
axes[1].set_title("Phase sample row counts")
axes[1].set_xlabel("rows")
lineage_fig = save_fig(fig, "cell12_lineage_inventory.png")

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
tp = target_profile[target_profile["dataset"].eq("D08_active")]
axes[0].bar(tp["target"], tp["n"], color="#6aa84f")
axes[0].set_title("D08 active target non-null N")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylabel("non-null rows")

split_summary.plot(kind="bar", x="split", y="school_n", ax=axes[1], legend=False, color="#9b7653")
axes[1].set_title("School-level split distribution")
axes[1].set_xlabel("split")
axes[1].set_ylabel("school count")
sample_target_fig = save_fig(fig, "cell12_sample_target_dashboard.png")

print("saved:", lineage_fig)
print("saved:", sample_target_fig)


saved: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5/figures/cell12_lineage_inventory.png
saved: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5/figures/cell12_sample_target_dashboard.png


## Cell 13. Visualization 03: Target Distributions


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
for ax, target in zip(axes, target_cols):
    vals = pd.to_numeric(d08[target], errors="coerce").dropna()
    ax.hist(vals, bins=40, color="#4f81bd", alpha=0.85, edgecolor="white")
    ax.axvline(vals.median(), color="#c44e52", linewidth=2, label=f"median={vals.median():.2f}")
    zero_n = int((vals == 0).sum())
    hundred_n = int((vals == 100).sum())
    ax.set_title(target)
    ax.set_xlabel("pct")
    ax.set_ylabel("row count")
    ax.text(0.02, 0.95, f"N={len(vals):,}\nzero={zero_n:,}\nhundred={hundred_n:,}", transform=ax.transAxes, va="top", fontsize=9)
    ax.legend()
target_dist_fig = save_fig(fig, "cell13_target_distributions.png")
print("saved:", target_dist_fig)


saved: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5/figures/cell13_target_distributions.png


## Cell 14. Visualization 04: Non-model Feature Signal Correlation


In [14]:
allowed_blocks = {"S0", "B", "Q", "C24", "GOMS", "POLICY"}
candidate_features = registry.loc[registry["feature_block"].isin(allowed_blocks), "column"].dropna().tolist()
numeric_features = []
for col in candidate_features:
    if col in d08.columns:
        vals = pd.to_numeric(d08[col], errors="coerce")
        if vals.notna().sum() >= 100 and vals.nunique(dropna=True) >= 3:
            numeric_features.append(col)

corr_rows = []
for target in target_cols:
    y = pd.to_numeric(d08[target], errors="coerce")
    for feature in numeric_features:
        x = pd.to_numeric(d08[feature], errors="coerce")
        ok = x.notna() & y.notna()
        if ok.sum() < 100:
            continue
        pearson = float(x[ok].corr(y[ok], method="pearson"))
        spearman = float(x[ok].corr(y[ok], method="spearman"))
        corr_rows.append({"target": target, "feature": feature, "n": int(ok.sum()), "pearson": pearson, "spearman": spearman, "abs_spearman": abs(spearman)})
corr_signal = pd.DataFrame(corr_rows).sort_values(["target", "abs_spearman"], ascending=[True, False])
atomic_csv(corr_signal, QA_DIR / "cell14_non_model_feature_signal_correlation.csv")
display(corr_signal.groupby("target").head(15))

fig, axes = plt.subplots(1, 3, figsize=(18, 7), constrained_layout=True)
for ax, target in zip(axes, target_cols):
    sub = corr_signal[corr_signal["target"].eq(target)].head(10).sort_values("spearman")
    ax.barh(sub["feature"], sub["spearman"], color=np.where(sub["spearman"] >= 0, "#6aa84f", "#c44e52"))
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Top Spearman signals: {target}")
    ax.set_xlabel("Spearman rho")
feature_signal_fig = save_fig(fig, "cell14_feature_signal_correlation.png")
print("saved:", feature_signal_fig)
print("Interpretation rule: correlation is association/hypothesis ranking only; no feature selection or causal claim.")


,target,feature,n,pearson,spearman,abs_spearman
0,a_rate_pct,selectivity_proxy_pct,3737,0.235223,0.299015,0.299015
8,a_rate_pct,fulltime_faculty_share_pct,6379,-0.160233,-0.165721,0.165721
7,a_rate_pct,student_faculty_ratio,6379,-0.082733,-0.151548,0.151548
3,a_rate_pct,admit_per_applicant_ratio,5260,-0.029132,-0.139333,0.139333
1,a_rate_pct,competition_ratio,5257,0.064665,0.133404,0.133404
5,a_rate_pct,female_student_share_pct,8441,0.128952,0.126311,0.126311
26,a_rate_pct,goms_recent_unstable_pct,10099,0.115361,0.112521,0.112521
25,a_rate_pct,goms_recent_permanent_pct,10099,-0.117012,-0.108910,0.108910
41,a_rate_pct,goms_latest_2019_permanent_pct,10099,-0.114048,-0.108910,0.108910
36,a_rate_pct,goms_firm_300plus_trend_per_year,10099,-0.099107,-0.105217,0.105217


saved: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5/figures/cell14_feature_signal_correlation.png
Interpretation rule: correlation is association/hypothesis ranking only; no feature selection or causal claim.


## Cell 15. Visualization 05: Existing P5/P6 Result Readout


In [15]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
p6_plot_metrics = [
    "P4_RAW_A_EMPLOYMENT_AME_PP",
    "P4_RAW_A_PROGRESSION_AME_PP",
    "P4_RESID_EMPLOYMENT_AME_PP",
    "P4_RESID_PROGRESSION_AME_PP",
]
vals = [p6_metric.get(m, np.nan) for m in p6_plot_metrics]
labels = [m.replace("P4_", "").replace("_AME_PP", "") for m in p6_plot_metrics]
axes[0].bar(labels, vals, color=["#8faadc", "#d9a441", "#8faadc", "#d9a441"])
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Existing P6 AME readout (percentage points)")
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_ylabel("pp")

p5_plot = top_major.sort_values("diff_pp")
axes[1].barh(p5_plot["major_group_7"], p5_plot["diff_pp"], color="#9b7653")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("P5 progression minus employment AME by major7")
axes[1].set_xlabel("percentage points")
result_fig = save_fig(fig, "cell15_result_readout.png")
print("saved:", result_fig)


saved: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5/figures/cell15_result_readout.png


## Cell 16. Final Status Matrix / Issue Register / Report / Manifest


In [16]:
issue_rows = []
def add_issue(issue_id: str, severity: str, evidence: str, impact: str, required_decision: str) -> None:
    issue_rows.append(
        {
            "issue_id": issue_id,
            "severity": severity,
            "evidence": evidence,
            "impact": impact,
            "required_decision": required_decision,
        }
    )

if "split" not in d08.columns:
    add_issue("WARN_D08_SPLIT_EXTERNAL", "WARN", "D08_active has no split column; split is in dim_school_split/membership", "Do not infer split from row order", "Use school_uid merge to dim_school_split")
if pct_range_violations(d08) == 0:
    add_issue("PASS_PCT_RANGE", "PASS", "0 pct range violations in active mart", "Non-blocking", "No decision")
else:
    add_issue("FAIL_PCT_RANGE", "FAIL", "pct range violation found", "Rate interpretation blocked", "Inspect offending rows")
if int(d08["outcome_row_id"].duplicated().sum()) == 0:
    add_issue("PASS_OUTCOME_ROW_ID", "PASS", "outcome_row_id duplicate rows = 0", "Stable row ID usable", "No decision")
add_issue("WARN_DIRECT_URL_LIMIT", "WARN", "Some source direct download URLs not in manifest; portal URL + local hash retained", "External provenance is portal-level for those files", "Keep local hashes as provenance anchor")
q_branch = p6_status.loc[p6_status["status_key"].eq("P6_Q_BRANCH_STATUS"), "status"]
if len(q_branch) and q_branch.iloc[0] != "READY":
    add_issue("WARN_Q_BRANCH_BLOCKED", "WARN", f"P6_Q_BRANCH_STATUS={q_branch.iloc[0]}", "Selectivity/Q branch is not launch-ready", "Resolve feature contract before Q branch interpretation")
if p5_status_json.get("P5_RESIDUAL_STATUS") == "PENDING_UPSTREAM_RESIDUAL":
    add_issue("WARN_P5_RESIDUAL_PENDING_IN_STRICT_STATUS", "WARN", "P5 v2 strict status says residual pending", "Do not cite P5 residual branch as complete from P5 status alone", "Use P6 run-up when citing later residual handoff")

issue_register = pd.DataFrame(issue_rows)
status_matrix = pd.DataFrame(
    [
        {"cell": "Cell03", "gate": "representative notebook clone", "status": "PASS", "evidence": "cell03_representative_notebook_clone_inventory.csv"},
        {"cell": "Cell04", "gate": "key file/source provenance", "status": "PASS", "evidence": "cell04_key_file_inventory.csv"},
        {"cell": "Cell05", "gate": "key/grain/range", "status": "PASS", "evidence": "cell05_mart_key_grain_range_integrity.csv"},
        {"cell": "Cell06", "gate": "schema/role registry", "status": "PASS", "evidence": "cell06_registry_audit.csv"},
        {"cell": "Cell07", "gate": "sample/split", "status": "PASS", "evidence": "cell07_phase_sample_registry.csv"},
        {"cell": "Cell08", "gate": "target distribution", "status": "PASS", "evidence": "cell08_target_distribution_profile.csv"},
        {"cell": "Cell09", "gate": "missing/context grain", "status": "WARN" if int(context_grain['status'].ne('PASS_MAJOR_CONTEXT_CONSTANT').sum()) else "PASS", "evidence": "cell09_context_grain_check.csv"},
        {"cell": "Cell10", "gate": "existing result readout", "status": "PASS", "evidence": "cell10_existing_numeric_result_readout.csv"},
        {"cell": "Cell11-15", "gate": "visualization artifacts", "status": "PASS", "evidence": "cell11_visualization_plan.csv + figures/*.png"},
    ]
)

atomic_csv(issue_register, QA_DIR / "cell16_issue_register.csv")
atomic_csv(status_matrix, QA_DIR / "cell16_status_matrix.csv")
display(status_matrix)
display(issue_register)

report = "\n".join(
    [
        "# P2 Cell-by-Cell Engineering QA Report",
        "",
        f"- run_id: `{RUN_ID}`",
        f"- final_status: `{'PASS_WITH_WARNINGS' if issue_register['severity'].eq('WARN').any() and not issue_register['severity'].eq('FAIL').any() else ('PASS' if not issue_register['severity'].eq('FAIL').any() else 'FAIL')}`",
        f"- active D08: `{D08_PATH.relative_to(PROJECT_ROOT)}` / `{d08.shape[0]} x {d08.shape[1]}` / `{sha256_file(D08_PATH)}`",
        f"- representative notebooks cloned: `{len(notebook_inventory)}`; hash mismatch: `{int((~notebook_inventory['clone_hash_match']).sum())}`",
        "",
        "## Status Matrix",
        md_table(status_matrix),
        "",
        "## Key Integrity",
        md_table(mart_integrity),
        "",
        "## Target Profile",
        md_table(target_profile),
        "",
        "## Visualization Plan",
        md_table(visual_plan),
        "",
        "## Interpretation",
        md_table(interpretation),
        "",
        "## Issues",
        md_table(issue_register),
    ]
)
report_path = REPORT_DIR / "CELL_BY_CELL_ENGINEERING_QA_REPORT.md"
atomic_text(report + "\n", report_path)

handoff = "\n".join(
    [
        "# HANDOFF TO CHATGPT - Cell-by-Cell Blueprint",
        "",
        f"FINAL_STATUS: {'PASS_WITH_WARNINGS' if issue_register['severity'].eq('WARN').any() and not issue_register['severity'].eq('FAIL').any() else ('PASS' if not issue_register['severity'].eq('FAIL').any() else 'FAIL')}",
        f"RUN_ID: {RUN_ID}",
        f"RUN_DIR: {RUN_DIR}",
        f"NOTEBOOK: {OUT_ROOT / 'P2_INTEGRATED_ENGINEERING_BLUEPRINT_CELL_BY_CELL.ipynb'}",
        f"ACTIVE_D08_SHAPE: {d08.shape[0]} x {d08.shape[1]}",
        f"ACTIVE_D08_SHA256: {sha256_file(D08_PATH)}",
        "STATUS_MATRIX:",
        md_table(status_matrix),
        "ISSUES:",
        md_table(issue_register),
        "VISUALS:",
        "- figures/cell12_lineage_inventory.png",
        "- figures/cell12_sample_target_dashboard.png",
        "- figures/cell13_target_distributions.png",
        "- figures/cell14_feature_signal_correlation.png",
        "- figures/cell15_result_readout.png",
    ]
)
atomic_text(handoff + "\n", RUN_DIR / "HANDOFF_TO_CHATGPT.md")

generated = [p for p in RUN_DIR.rglob("*") if p.is_file()]
manifest_rows = []
for p in sorted(generated):
    shape, loader, error = shape_string(p)
    manifest_rows.append(
        {
            "relative_path": str(p.relative_to(PROJECT_ROOT)),
            "size_bytes": p.stat().st_size,
            "shape": shape,
            "loader": loader,
            "sha256": sha256_file(p),
        }
    )
manifest = pd.DataFrame(manifest_rows)
atomic_csv(manifest, RUN_DIR / "CELL_BY_CELL_MANIFEST.csv")
atomic_text(json.dumps(manifest_rows, ensure_ascii=False, indent=2), RUN_DIR / "CELL_BY_CELL_MANIFEST.json")

print("CELL16 COMPLETE")
print("RUN_DIR:", RUN_DIR)
print("REPORT:", report_path)
print("MANIFEST:", RUN_DIR / "CELL_BY_CELL_MANIFEST.csv")
print("FIGURES:", sorted(str(p.name) for p in FIG_DIR.glob("*.png")))


,cell,gate,status,evidence
0,Cell03,representative notebook clone,PASS,cell03_representative_notebook_clone_inventory...
1,Cell04,key file/source provenance,PASS,cell04_key_file_inventory.csv
2,Cell05,key/grain/range,PASS,cell05_mart_key_grain_range_integrity.csv
3,Cell06,schema/role registry,PASS,cell06_registry_audit.csv
4,Cell07,sample/split,PASS,cell07_phase_sample_registry.csv
5,Cell08,target distribution,PASS,cell08_target_distribution_profile.csv
6,Cell09,missing/context grain,PASS,cell09_context_grain_check.csv
7,Cell10,existing result readout,PASS,cell10_existing_numeric_result_readout.csv
8,Cell11-15,visualization artifacts,PASS,cell11_visualization_plan.csv + figures/*.png


,issue_id,severity,evidence,impact,required_decision
0,WARN_D08_SPLIT_EXTERNAL,WARN,D08_active has no split column; split is in di...,Do not infer split from row order,Use school_uid merge to dim_school_split
1,PASS_PCT_RANGE,PASS,0 pct range violations in active mart,Non-blocking,No decision
2,PASS_OUTCOME_ROW_ID,PASS,outcome_row_id duplicate rows = 0,Stable row ID usable,No decision
3,WARN_DIRECT_URL_LIMIT,WARN,Some source direct download URLs not in manife...,External provenance is portal-level for those ...,Keep local hashes as provenance anchor
4,WARN_Q_BRANCH_BLOCKED,WARN,P6_Q_BRANCH_STATUS=BLOCKED_FEATURE_CONTRACT,Selectivity/Q branch is not launch-ready,Resolve feature contract before Q branch inter...
5,WARN_P5_RESIDUAL_PENDING_IN_STRICT_STATUS,WARN,P5 v2 strict status says residual pending,Do not cite P5 residual branch as complete fro...,Use P6 run-up when citing later residual handoff


CELL16 COMPLETE
RUN_DIR: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5
REPORT: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5/reports/CELL_BY_CELL_ENGINEERING_QA_REPORT.md
MANIFEST: /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_integrated_engineering_blueprint_v1/cell_runs/20260713T091641Z_5b1a3d5/CELL_BY_CELL_MANIFEST.csv
FIGURES: ['cell12_lineage_inventory.png', 'cell12_sample_target_dashboard.png', 'cell13_target_distributions.png', 'cell14_feature_signal_correlation.png', 'cell15_result_readout.png']
